# Brazilian Marketplace (Olist) — Business Analysis

Olist is a Brazilian marketplace connecting small businesses to a wide network of online sales channels. This notebook analyzes ~100,000 orders placed on the Olist Store between 2016 and 2018, using the public [Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0).

**Business question:** How can Olist optimize its operational and marketing strategy to improve profitability, increase customer satisfaction, and reduce delivery times, based on historical sales data, customer reviews, and logistics metrics?

**Method:** SQL on Databricks — data modeling, cleaning, revenue analysis, AOV, CLV, RFM segmentation, churn rate, and correlation studies.

## 1. Setup & Schema Overview

The 9 source CSVs were loaded as tables in `workspace.brazilian_marketplace`. Before analyzing, let's confirm the schema of each table matches the dataset documentation.

In [0]:
-- I'm checking the schema of every table I loaded, to make sure it matches what I expect from the dataset documentation before I do anything else with it

SELECT table_name, column_name, data_type, ordinal_position
FROM workspace.information_schema.columns
WHERE table_schema = 'brazilian_marketplace'
ORDER BY table_name, ordinal_position;

The schema matches the dataset documentation for 7 of 9 tables. Two exceptions worth noting:

- `product_category_name_translation` carries an extra `dummy_column` (a constant value on every row) — a leftover from the original BigQuery load, which needed at least one numeric-looking column. Excluded from all queries going forward.
- `reviews` only has `review_id`, `order_id`, `review_score` (not the full 7-column review dataset) — we're using the pre-cleaned version of the reviews file, which already dropped free-text comments and dates. Sufficient here, since this analysis only needs the review score.

Row counts per table, as a sanity check against the documented dataset size:


In [0]:
-- Quick sanity check: do my row counts match what the Olist dataset documentation says each table should have?

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM workspace.brazilian_marketplace.customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM workspace.brazilian_marketplace.geolocation
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.brazilian_marketplace.order_items
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.brazilian_marketplace.orders
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.brazilian_marketplace.payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.brazilian_marketplace.reviews
UNION ALL SELECT 'products', COUNT(*) FROM workspace.brazilian_marketplace.products
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.brazilian_marketplace.sellers
UNION ALL SELECT 'product_category_name_translation', COUNT(*) FROM workspace.brazilian_marketplace.product_category_name_translation;

All counts match the documented Olist dataset sizes (e.g. 99,441 customers and orders, 112,650 order items, 1,000,163 geolocation records) — the load is clean.

## 2. Primary Key Identification

For each table, a column is a primary key candidate if `COUNT(*) = COUNT(DISTINCT column)` — every value is present and unique.

In [0]:
-- For each table, I'm testing whether a column could be a primary key: if COUNT(*) equals COUNT(DISTINCT column), every value is unique and present, so it's a valid single-column key

SELECT 'customers' AS table_name, 'customer_id' AS candidate_column,
       COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_non_null,
       COUNT(*) = COUNT(DISTINCT customer_id) AS is_primary_key
FROM workspace.brazilian_marketplace.customers
UNION ALL
SELECT 'orders', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.orders
UNION ALL
SELECT 'products', 'product_id', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) = COUNT(DISTINCT product_id)
FROM workspace.brazilian_marketplace.products
UNION ALL
SELECT 'sellers', 'seller_id', COUNT(*), COUNT(DISTINCT seller_id), COUNT(*) = COUNT(DISTINCT seller_id)
FROM workspace.brazilian_marketplace.sellers
UNION ALL
SELECT 'reviews', 'review_id', COUNT(*), COUNT(DISTINCT review_id), COUNT(*) = COUNT(DISTINCT review_id)
FROM workspace.brazilian_marketplace.reviews
UNION ALL
SELECT 'product_category_name_translation', 'product_category_name', COUNT(*), COUNT(DISTINCT product_category_name), COUNT(*) = COUNT(DISTINCT product_category_name)
FROM workspace.brazilian_marketplace.product_category_name_translation
UNION ALL
SELECT 'order_items', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'geolocation', 'geolocation_zip_code_prefix', COUNT(*), COUNT(DISTINCT geolocation_zip_code_prefix), COUNT(*) = COUNT(DISTINCT geolocation_zip_code_prefix)
FROM workspace.brazilian_marketplace.geolocation;

Four tables came back `false` above. For `order_items`, `payments`, and `geolocation` this is expected (an order can have several items or payment installments; a zip prefix covers many lat/lng points). But `reviews.review_id` repeating is unexpected — a review should be unique. Let's look closer before settling on a composite key.


In [0]:
-- I want to see which review_ids repeat and how often, to understand why review_id alone isn't unique
SELECT review_id, COUNT(*) AS n_occurrences
FROM workspace.brazilian_marketplace.reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY n_occurrences DESC
LIMIT 10;

In [0]:
-- Let's look at the actual rows behind one of these duplicated review_ids, to see what's different between the copies
SELECT *
FROM workspace.brazilian_marketplace.reviews
WHERE review_id IN (
  SELECT review_id
  FROM workspace.brazilian_marketplace.reviews
  GROUP BY review_id
  HAVING COUNT(*) > 1
)
ORDER BY review_id
LIMIT 20;

The duplicates differ by `order_id`: Olist occasionally ties one review survey to more than one order (e.g. orders placed close together get bundled into a single satisfaction survey). So `review_id` alone isn't a key, but `review_id` + `order_id` together should be. Let's confirm that, along with the other composite keys.


In [0]:
-- Checking composite keys for the three tables that failed the single-column test: order_id + order_item_id for order_items, order_id + payment_sequential for payments, review_id + order_id for reviews
SELECT 'order_items' AS table_name, COUNT(*) AS total_rows,
       COUNT(DISTINCT CONCAT(order_id, '-', order_item_id)) AS distinct_composite
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', COUNT(*), COUNT(DISTINCT CONCAT(order_id, '-', payment_sequential))
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT CONCAT(review_id, '-', order_id))
FROM workspace.brazilian_marketplace.reviews;


## Primary Key Summary

| Table | Primary key |
|---|---|
| customers | `customer_id` |
| orders | `order_id` |
| products | `product_id` |
| sellers | `seller_id` |
| product_category_name_translation | `product_category_name` |
| order_items | `order_id` + `order_item_id` (composite) |
| payments | `order_id` + `payment_sequential` (composite) |
| reviews | `review_id` + `order_id` (composite) |
| geolocation | no single natural key — it's a reference lookup table (many lat/lng points share a zip prefix); not decomposed further, since it's only ever joined on `geolocation_zip_code_prefix` |

## 3. ER Schema / Data Model

Relationships between the tables:

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : places
    ORDERS ||--o{ ORDER_ITEMS : contains
    ORDERS ||--o{ PAYMENTS : "paid via"
    ORDERS ||--o{ REVIEWS : receives
    ORDER_ITEMS }o--|| PRODUCTS : references
    ORDER_ITEMS }o--|| SELLERS : "sold by"
    PRODUCTS }o--|| PRODUCT_CATEGORY_NAME_TRANSLATION : "category in"
    CUSTOMERS }o--|| GEOLOCATION : "zip code in"
    SELLERS }o--|| GEOLOCATION : "zip code in"
```